# Week 77, Day 4: Authentication

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w077d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('miniflask.py', 'w', encoding='utf-8') as f:
    f.write('"""A small subset of Flask that runs in the page (Week 77).\n\nThe names and behaviour follow Flask\'s own, so code written against this module also runs with\nthe real Flask (pip install flask): Flask, request, g, jsonify, abort, redirect, make_response,\nurl_for, Blueprint, and app.test_client(). It\'s a teaching copy, not the real thing: no templates,\nsessions, streaming or debugger.\n"""\nimport io\nimport json\nimport re\nfrom contextvars import ContextVar\nfrom http import HTTPStatus\nfrom urllib.parse import parse_qs, urlencode, urlsplit\n\n_current = ContextVar("request")\n\n\nclass HTTPException(Exception):\n    def __init__(self, code, description=None):\n        self.code = code\n        self.name = HTTPStatus(code).phrase\n        self.description = description or self.name\n        super().__init__(f"{code} {self.name}: {self.description}")\n\n\ndef abort(code, description=None):\n    raise HTTPException(code, description)\n\n\nclass Headers:\n    """Case-insensitive headers."""\n    def __init__(self, items=()):\n        self._items = [(k, str(v)) for k, v in (items.items() if isinstance(items, dict) else items)]\n\n    def get(self, name, default=None):\n        for k, v in self._items:\n            if k.lower() == name.lower():\n                return v\n        return default\n\n    def __getitem__(self, name):\n        value = self.get(name)\n        if value is None:\n            raise KeyError(name)\n        return value\n\n    def __contains__(self, name):\n        return self.get(name) is not None\n\n    def __setitem__(self, name, value):\n        self._items = [(k, v) for k, v in self._items if k.lower() != name.lower()] + [(name, str(value))]\n\n    def items(self):\n        return list(self._items)\n\n\nclass Args(dict):\n    """Query or form values: .get(name) gives the first value; .getlist(name) all of them."""\n    def __init__(self, pairs):\n        super().__init__()\n        self._lists = {}\n        for k, v in pairs:\n            self._lists.setdefault(k, []).append(v)\n            dict.setdefault(self, k, v)\n\n    def get(self, key, default=None, type=None):\n        if key not in self:\n            return default\n        value = self[key]\n        if type is not None:\n            try:\n                return type(value)\n            except (TypeError, ValueError):\n                return default\n        return value\n\n    def getlist(self, key):\n        return list(self._lists.get(key, []))\n\n\nclass Request:\n    def __init__(self, environ):\n        self.environ = environ\n        self.method = environ["REQUEST_METHOD"].upper()\n        self.path = environ.get("PATH_INFO", "/") or "/"\n        self.args = Args(_pairs(environ.get("QUERY_STRING", "")))\n        items = [(k[5:].replace("_", "-").title(), v) for k, v in environ.items() if k.startswith("HTTP_")]\n        if environ.get("CONTENT_TYPE"):\n            items.append(("Content-Type", environ["CONTENT_TYPE"]))\n        self.headers = Headers(items)\n        length = int(environ.get("CONTENT_LENGTH") or 0)\n        self.data = environ["wsgi.input"].read(length) if length else b""\n        self.content_type = environ.get("CONTENT_TYPE", "")\n        self.form = Args(_pairs(self.data.decode("utf-8"))) if self.content_type.startswith("application/x-www-form-urlencoded") else Args([])\n        self.view_args = {}\n        cookie = environ.get("HTTP_COOKIE", "")\n        self.cookies = dict(p.strip().split("=", 1) for p in cookie.split(";") if "=" in p)\n\n    @property\n    def is_json(self):\n        return self.content_type.split(";")[0].strip() == "application/json"\n\n    def get_json(self, silent=False, force=False):\n        if not (self.is_json or force):\n            if silent:\n                return None\n            abort(415, "expected a JSON body (Content-Type: application/json)")\n        try:\n            return json.loads(self.data or b"null")\n        except ValueError:\n            if silent:\n                return None\n            abort(400, "the body isn\'t valid JSON")\n\n    @property\n    def json(self):\n        return self.get_json()\n\n\ndef _pairs(text):\n    return [(k, v) for k, vs in parse_qs(text, keep_blank_values=True).items() for v in vs]\n\n\nclass _Proxy:\n    def __init__(self, getter):\n        object.__setattr__(self, "_getter", getter)\n\n    def __getattr__(self, name):\n        return getattr(self._getter(), name)\n\n    def __setattr__(self, name, value):\n        setattr(self._getter(), name, value)\n\n\ndef _request():\n    try:\n        return _current.get()[0]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\ndef _g():\n    try:\n        return _current.get()[1]\n    except LookupError:\n        raise RuntimeError("working outside of request context") from None\n\n\nclass _G:\n    def get(self, name, default=None):\n        return self.__dict__.get(name, default)\n\n\nrequest = _Proxy(_request)\ng = _Proxy(_g)\n\n\nclass Response:\n    def __init__(self, response=b"", status=200, headers=None, mimetype=None, content_type=None):\n        if isinstance(response, str):\n            response = response.encode("utf-8")\n        self.data = response\n        self.status_code = status\n        self.headers = Headers(headers or {})\n        if content_type or mimetype or "Content-Type" not in self.headers:\n            ctype = content_type or (f"{mimetype}; charset=utf-8" if mimetype and mimetype.startswith("text/") else mimetype) or "text/html; charset=utf-8"\n            self.headers["Content-Type"] = ctype\n\n    @property\n    def status(self):\n        return f"{self.status_code} {HTTPStatus(self.status_code).phrase}"\n\n    @property\n    def text(self):\n        return self.data.decode("utf-8")\n\n    def get_json(self):\n        return json.loads(self.data)\n\n    @property\n    def json(self):\n        return self.get_json()\n\n    def set_cookie(self, key, value="", max_age=None, path="/", secure=False, httponly=False, samesite=None):\n        parts = [f"{key}={value}"]\n        if max_age is not None:\n            parts.append(f"Max-Age={max_age}")\n        parts.append(f"Path={path}")\n        if secure:\n            parts.append("Secure")\n        if httponly:\n            parts.append("HttpOnly")\n        if samesite:\n            parts.append(f"SameSite={samesite}")\n        self.headers._items.append(("Set-Cookie", "; ".join(parts)))\n\n\ndef jsonify(*args, **kwargs):\n    data = args[0] if len(args) == 1 else (list(args) if args else kwargs)\n    return Response(json.dumps(data), headers={"Content-Type": "application/json"})\n\n\ndef redirect(location, code=302):\n    return Response(b"", code, {"Location": location})\n\n\ndef make_response(*args):\n    rv = args[0] if len(args) == 1 else args\n    return _App._to_response(rv)\n\n\ndef url_for(endpoint, **values):\n    return _current.get()[2].url_for(endpoint, **values)\n\n\n_CONVERTERS = {"int": (r"\\d+", int), "float": (r"\\d+(?:\\.\\d+)?", float), "path": (r".+", str), "string": (r"[^/]+", str)}\n\n\nclass _Rule:\n    def __init__(self, rule, endpoint, view, methods):\n        self.rule, self.endpoint, self.view = rule, endpoint, view\n        self.methods = {m.upper() for m in methods} | ({"HEAD"} if "GET" in {m.upper() for m in methods} else set())\n        regex, self.converters = "", {}\n        for piece in re.split(r"(<[^>]+>)", rule):\n            m = re.fullmatch(r"<(?:(\\w+):)?(\\w+)>", piece)\n            if m:\n                kind, name = m.group(1) or "string", m.group(2)\n                pattern, convert = _CONVERTERS[kind]\n                regex += f"(?P<{name}>{pattern})"\n                self.converters[name] = convert\n            else:\n                regex += re.escape(piece)\n        self.regex = re.compile(regex)\n\n    def match(self, path):\n        m = self.regex.fullmatch(path)\n        if not m:\n            return None\n        return {k: self.converters[k](v) for k, v in m.groupdict().items()}\n\n    def build(self, values):\n        def fill(m):\n            return str(values.pop(m.group(2)))\n        path = re.sub(r"<(?:(\\w+):)?(\\w+)>", fill, self.rule)\n        return path + (f"?{urlencode(values)}" if values else "")\n\n\nclass _Setup:\n    def route(self, rule, methods=("GET",), endpoint=None):\n        def decorate(view):\n            self._add(rule, endpoint or view.__name__, view, methods)\n            return view\n        return decorate\n\n    def get(self, rule, **kw):\n        return self.route(rule, methods=["GET"], **kw)\n\n    def post(self, rule, **kw):\n        return self.route(rule, methods=["POST"], **kw)\n\n    def put(self, rule, **kw):\n        return self.route(rule, methods=["PUT"], **kw)\n\n    def patch(self, rule, **kw):\n        return self.route(rule, methods=["PATCH"], **kw)\n\n    def delete(self, rule, **kw):\n        return self.route(rule, methods=["DELETE"], **kw)\n\n\nclass Blueprint(_Setup):\n    def __init__(self, name, import_name, url_prefix=""):\n        self.name, self.url_prefix = name, url_prefix\n        self.deferred = []\n\n    def _add(self, rule, endpoint, view, methods):\n        self.deferred.append((rule, endpoint, view, methods))\n\n\nclass _App(_Setup):\n    def __init__(self, import_name):\n        self.import_name = import_name\n        self.config = {"TESTING": False}\n        self.rules = []\n        self.before_funcs, self.after_funcs, self.error_handlers = [], [], {}\n\n    def _add(self, rule, endpoint, view, methods):\n        self.rules.append(_Rule(rule, endpoint, view, methods))\n\n    def register_blueprint(self, bp, url_prefix=None):\n        prefix = url_prefix if url_prefix is not None else bp.url_prefix\n        for rule, endpoint, view, methods in bp.deferred:\n            self._add(prefix + rule, f"{bp.name}.{endpoint}", view, methods)\n\n    def before_request(self, f):\n        self.before_funcs.append(f)\n        return f\n\n    def after_request(self, f):\n        self.after_funcs.append(f)\n        return f\n\n    def errorhandler(self, code):\n        def decorate(f):\n            self.error_handlers[code] = f\n            return f\n        return decorate\n\n    def url_for(self, endpoint, **values):\n        for r in self.rules:\n            if r.endpoint == endpoint:\n                return r.build(dict(values))\n        raise LookupError(f"no endpoint {endpoint!r}")\n\n    @staticmethod\n    def _to_response(rv):\n        status, headers = None, None\n        if isinstance(rv, tuple):\n            if len(rv) == 3:\n                rv, status, headers = rv\n            elif isinstance(rv[1], (dict, list)):\n                rv, headers = rv\n            else:\n                rv, status = rv\n        if isinstance(rv, Response):\n            response = rv\n        elif isinstance(rv, (dict, list)):\n            response = jsonify(rv)\n        elif isinstance(rv, (str, bytes)):\n            response = Response(rv)\n        elif rv is None:\n            raise TypeError("the view function did not return a valid response")\n        else:\n            raise TypeError(f"can\'t turn {type(rv).__name__} into a response")\n        if status is not None:\n            response.status_code = status\n        for k, v in (headers.items() if isinstance(headers, dict) else headers or []):\n            response.headers[k] = v\n        return response\n\n    def _error(self, e):\n        handler = self.error_handlers.get(e.code)\n        if handler:\n            return self._to_response(handler(e))\n        body = f"<h1>{e.code} {e.name}</h1><p>{e.description}</p>"\n        return Response(body, e.code)\n\n    def _dispatch(self, req):\n        path = req.path\n        allowed = set()\n        for r in self.rules:\n            values = r.match(path)\n            if values is None:\n                continue\n            if req.method in r.methods:\n                req.view_args = values\n                return r.view(**values)\n            allowed |= r.methods\n        if allowed:\n            raise _MethodNotAllowed(sorted(allowed | {"OPTIONS"}))\n        abort(404, "The requested URL was not found on the server.")\n\n    def full_dispatch(self, environ):\n        req = Request(environ)\n        token = _current.set((req, _G(), self))\n        try:\n            try:\n                rv = None\n                for f in self.before_funcs:\n                    rv = f()\n                    if rv is not None:\n                        break\n                if rv is None:\n                    rv = self._dispatch(req)\n                response = self._to_response(rv)\n            except _MethodNotAllowed as e:\n                response = self._error(HTTPException(405))\n                response.headers["Allow"] = ", ".join(e.allowed)\n            except HTTPException as e:\n                response = self._error(e)\n            except Exception as e:\n                if self.config.get("TESTING") and 500 not in self.error_handlers:\n                    raise\n                response = self._error(HTTPException(500)) if 500 not in self.error_handlers else self._to_response(self.error_handlers[500](e))\n                response.status_code = 500\n            for f in self.after_funcs:\n                response = f(response) or response\n            if req.method == "HEAD":\n                response.headers["Content-Length"] = len(response.data)\n                response.data = b""\n            return response\n        finally:\n            _current.reset(token)\n\n    def __call__(self, environ, start_response):\n        response = self.full_dispatch(environ)\n        if "Content-Length" not in response.headers:\n            response.headers["Content-Length"] = len(response.data)\n        start_response(response.status, response.headers.items())\n        return [response.data]\n\n    def test_client(self):\n        return _TestClient(self)\n\n\nclass _MethodNotAllowed(Exception):\n    def __init__(self, allowed):\n        self.allowed = allowed\n\n\nclass Flask(_App):\n    pass\n\n\nclass _TestClient:\n    def __init__(self, app):\n        self.app = app\n        self.cookies = {}\n\n    def open(self, path, method="GET", json=None, data=None, headers=None, query_string=None):\n        url = urlsplit(path)\n        query = url.query\n        if query_string:\n            query = (query + "&" if query else "") + urlencode(query_string, doseq=True)\n        body, ctype = b"", None\n        if json is not None:\n            import json as _json\n            body, ctype = _json.dumps(json).encode("utf-8"), "application/json"\n        elif isinstance(data, dict):\n            body, ctype = urlencode(data, doseq=True).encode("utf-8"), "application/x-www-form-urlencoded"\n        elif data is not None:\n            body = data.encode("utf-8") if isinstance(data, str) else data\n        environ = {"REQUEST_METHOD": method.upper(), "PATH_INFO": url.path or "/", "QUERY_STRING": query,\n                   "SERVER_NAME": "localhost", "SERVER_PORT": "80", "wsgi.url_scheme": "http",\n                   "wsgi.input": io.BytesIO(body), "CONTENT_LENGTH": str(len(body))}\n        headers = dict(headers or {})\n        if ctype and not any(k.lower() == "content-type" for k in headers):\n            environ["CONTENT_TYPE"] = ctype\n        for k, v in headers.items():\n            if k.lower() == "content-type":\n                environ["CONTENT_TYPE"] = v\n            else:\n                environ["HTTP_" + k.upper().replace("-", "_")] = v\n        if self.cookies:\n            environ["HTTP_COOKIE"] = "; ".join(f"{k}={v}" for k, v in self.cookies.items())\n        response = self.app.full_dispatch(environ)\n        for k, v in response.headers.items():\n            if k.lower() == "set-cookie":\n                name, _, rest = v.partition("=")\n                self.cookies[name] = rest.split(";")[0]\n        return response\n\n    def get(self, path, **kw):\n        return self.open(path, "GET", **kw)\n\n    def post(self, path, **kw):\n        return self.open(path, "POST", **kw)\n\n    def put(self, path, **kw):\n        return self.open(path, "PUT", **kw)\n\n    def patch(self, path, **kw):\n        return self.open(path, "PATCH", **kw)\n\n    def delete(self, path, **kw):\n        return self.open(path, "DELETE", **kw)\n')

**Predict before you run:** what will this print?



In [ ]:
import hashlib

users = {"ada": "letmein", "bob": "letmein", "cy": "Tr0ub4dor&3"}
stored = {name: hashlib.sha256(pw.encode()).hexdigest() for name, pw in users.items()}
print(stored["ada"] == stored["bob"])

common = ["123456", "password", "letmein", "qwerty"]
lookup = {hashlib.sha256(pw.encode()).hexdigest(): pw for pw in common}
print({name: lookup.get(h, "?") for name, h in stored.items()})

In [ ]:
import base64, hashlib, hmac, secrets

def pbkdf2_sha256(password, salt, iterations, length=32):
    out = b""
    block = 1
    while len(out) < length:
        u = hmac.new(password, salt + block.to_bytes(4, "big"), hashlib.sha256).digest()
        total = int.from_bytes(u, "big")
        for _ in range(iterations - 1):
            u = hmac.new(password, u, hashlib.sha256).digest()
            total ^= int.from_bytes(u, "big")
        out += total.to_bytes(32, "big")
        block += 1
    return out[:length]

print(pbkdf2_sha256(b"passwd", b"salt", 1, 16).hex())

salt = secrets.token_bytes(16)
stored = pbkdf2_sha256(b"letmein", salt, 1000)
attempt = pbkdf2_sha256(b"letmein", salt, 1000)
print(hmac.compare_digest(stored, attempt), base64.b64encode(salt).decode() != "", len(stored))

In [ ]:
import base64, hashlib, hmac, json

def b64(data):
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode()

def jwt(payload, key):
    header = b64(json.dumps({"alg": "HS256", "typ": "JWT"}, separators=(",", ":")).encode())
    body = b64(json.dumps(payload, separators=(",", ":")).encode())
    signature = hmac.new(key, f"{header}.{body}".encode(), hashlib.sha256).digest()
    return f"{header}.{body}.{b64(signature)}"

token = jwt({"sub": "42", "role": "reader", "exp": 1767225600}, b"server-secret-key")
print(token[:40], "...", token.count("."))
payload_part = token.split(".")[1]
print(json.loads(base64.urlsafe_b64decode(payload_part + "=" * (-len(payload_part) % 4))))

> **Key idea.** Never store passwords: store a per-user random salt and a slow key-derivation result (PBKDF2, scrypt or Argon2) with its settings, and compare with hmac.compare_digest. Keep users logged in with random session ids in HttpOnly, Secure cookies (deleted on logout), cookies signed with an HMAC, or JWTs: base64url header, payload and HMAC signature, which anyone can read but only the key holder can make; verify the signature in constant time, insist on the expected algorithm (never none), and check exp and nbf; keep them short-lived with refresh tokens. Authorise with permissions granted by roles plus ownership checks on every object (broken object-level authorisation is the top API flaw), deny by default, and enforce in one place. Rate-limit and lock logins, offer MFA, give one error for unknown users and wrong passwords, and use OAuth/OpenID Connect libraries for third-party sign-in.

### Exercise: PBKDF2 by hand

Write `pbkdf2_sha256(password, salt, iterations, length=32)` (both bytes): PBKDF2 with HMAC-SHA256, as described in the lesson: output blocks numbered from 1, each `U1 = HMAC(password, salt + block number as 4 big-endian bytes)`, then `Uj = HMAC(password, U(j-1))` up to `iterations`, XORed together; concatenate blocks and cut to `length`. Raise `ValueError` for fewer than 1 iteration. Its results must match the standard ones (the checks compare known answers).

In [ ]:
import hashlib, hmac

def pbkdf2_sha256(password, salt, iterations, length=32):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pbkdf2():\n    \"matches the standard\"\n    assert pbkdf2_sha256(b\"passwd\", b\"salt\", 1, 64).hex() == \"55ac046e56e3089fec1691c22544b605f94185216dde0465e68b9d57c20dacbc49ca9cccf179b645991664b39d77ef317c71b845b1e30bd509112041d3a19783\"\n    assert pbkdf2_sha256(b\"correct horse\", b\"NaCl-1234567890\", 1000).hex() == \"606ffd49ce1f9bc147b2f3a1f63eb69ec385c3ad3a5be9493e673a328f860a5b\"\n    assert pbkdf2_sha256(b\"pw\", b\"saltsalt\", 2, 40).hex() == \"c73ec6e5a803119806c71b6adb7f1435d7a7030597c9b7223bb8541f7067be85e3813fcdf2f12e2f\"\n    try:\n        pbkdf2_sha256(b\"pw\", b\"s\", 0)\n        assert False\n    except ValueError:\n        pass")

### Exercise: Storing and checking passwords

Using `pbkdf2_sha256` (given), write `hash_password(password, iterations=1000, salt=None)`: a string `pbkdf2_sha256$ITERATIONS$SALT$HASH` with a fresh 16-byte random salt from `secrets` (unless one is given), and salt and hash in standard base64. Then `verify_password(password, stored)`: recompute with the stored settings and compare with `hmac.compare_digest`; a malformed string or another algorithm returns `False`, never raises. Finally `needs_rehash(stored, iterations)`: true if the stored iteration count is below `iterations`.

In [ ]:
import base64, hashlib, hmac, secrets

def pbkdf2_sha256(password, salt, iterations, length=32):
    out, block = b"", 1
    while len(out) < length:
        u = hmac.new(password, salt + block.to_bytes(4, "big"), hashlib.sha256).digest()
        total = int.from_bytes(u, "big")
        for _ in range(iterations - 1):
            u = hmac.new(password, u, hashlib.sha256).digest()
            total ^= int.from_bytes(u, "big")
        out += total.to_bytes(32, "big")
        block += 1
    return out[:length]

def hash_password(password, iterations=1000, salt=None):
    ___

def verify_password(password, stored):
    ___

def needs_rehash(stored, iterations):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_storage():\n    \"salted, slow, constant-time\"\n    a, b = hash_password(\"letmein\"), hash_password(\"letmein\")\n    assert a != b and a.startswith(\"pbkdf2_sha256$1000$\") and a.count(\"$\") == 3\n    assert verify_password(\"letmein\", a) and not verify_password(\"letmeout\", a)\n    fixed = hash_password(\"pw\", iterations=2, salt=b\"saltsalt\")\n    assert fixed == \"pbkdf2_sha256$2$c2FsdHNhbHQ=$xz7G5agDEZgGxxtq238UNden\" + fixed.split(\"$\")[3][24:] and verify_password(\"pw\", fixed)\n    for bad in [\"nonsense\", \"md5$1$x$y\", \"pbkdf2_sha256$abc$x$y\", \"pbkdf2_sha256$1000$!!$!!\"]:\n        assert verify_password(\"pw\", bad) is False\n    assert needs_rehash(a, 600000) and not needs_rehash(a, 1000)\n    assert \"compare_digest\" in source")

### Exercise: Signed cookies

Write `sign(data, key, now)`: a cookie value for a dict `data`: the base64url (no padding) of the JSON `{"data": data, "at": now}`, a `.`, and the base64url of its HMAC-SHA256 with `key`. Then `unsign(value, key, now, max_age)`: the data if the signature is right (compared with `compare_digest`) and it was signed no more than `max_age` seconds ago; raise `ValueError("bad signature")` for a tampered or malformed value and `ValueError("expired")` for an old one.

In [ ]:
import base64, hashlib, hmac, json

def sign(data, key, now):
    ___

def unsign(value, key, now, max_age):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_signed():\n    \"readable but not forgeable\"\n    import base64, json\n    key = b\"k\" * 32\n    value = sign({\"user\": 42, \"theme\": \"dark\"}, key, now=1000)\n    assert unsign(value, key, now=1500, max_age=3600) == {\"user\": 42, \"theme\": \"dark\"}\n    body = value.split(\".\")[0]\n    assert json.loads(base64.urlsafe_b64decode(body + \"=\" * (-len(body) % 4)))[\"data\"][\"user\"] == 42\n    forged = base64.urlsafe_b64encode(json.dumps({\"data\": {\"user\": 1}, \"at\": 1000}).encode()).decode().rstrip(\"=\") + \".\" + value.split(\".\")[1]\n    for bad in [forged, value + \"x\", \"nonsense\", value.replace(\".\", \"\")]:\n        try:\n            unsign(bad, key, now=1500, max_age=3600)\n            assert False, bad\n        except ValueError as e:\n            assert str(e) == \"bad signature\"\n    try:\n        unsign(value, b\"other key\", now=1500, max_age=3600)\n        assert False\n    except ValueError:\n        pass\n    try:\n        unsign(value, key, now=5000, max_age=3600)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"expired\"")

### Exercise: JSON Web Tokens

Write `encode_jwt(payload, key)`: a compact HS256 JWT (header `{"alg":"HS256","typ":"JWT"}`, compact JSON with `separators=(",", ":")`, base64url without padding). Then `decode_jwt(token, key, now, leeway=0)`: verify and return the payload, raising `InvalidToken` (given) with the message `"malformed"` (not three parts, or undecodable), `"algorithm not allowed"` (the header's `alg` isn't `HS256`, including `none`), `"bad signature"`, `"expired"` (when `now > exp + leeway`) or `"not yet valid"` (when `now < nbf - leeway`); `exp` and `nbf` are optional.

In [ ]:
import base64, hashlib, hmac, json

class InvalidToken(Exception):
    pass

def encode_jwt(payload, key):
    ___

def decode_jwt(token, key, now, leeway=0):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_jwt():\n    \"signatures, algorithms and times\"\n    import base64, json\n    key = b\"secret-key-for-tests\"\n    token = encode_jwt({\"sub\": \"42\", \"role\": \"editor\", \"exp\": 2000, \"nbf\": 1000}, key)\n    assert token.startswith(\"eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.\") and token.count(\".\") == 2\n    assert decode_jwt(token, key, now=1500) == {\"sub\": \"42\", \"role\": \"editor\", \"exp\": 2000, \"nbf\": 1000}\n    def b64(d):\n        return base64.urlsafe_b64encode(json.dumps(d, separators=(\",\", \":\")).encode()).decode().rstrip(\"=\")\n    none_alg = b64({\"alg\": \"none\", \"typ\": \"JWT\"}) + \".\" + b64({\"sub\": \"1\", \"role\": \"admin\"}) + \".\"\n    tampered = token.split(\".\")[0] + \".\" + b64({\"sub\": \"42\", \"role\": \"admin\", \"exp\": 2000, \"nbf\": 1000}) + \".\" + token.split(\".\")[2]\n    cases = [(none_alg, 1500, \"algorithm not allowed\"), (tampered, 1500, \"bad signature\"), (token, 2001, \"expired\"),\n             (token, 999, \"not yet valid\"), (\"a.b\", 1500, \"malformed\"), (\"!!.!!.!!\", 1500, \"malformed\")]\n    for t, now, message in cases:\n        try:\n            decode_jwt(t, key, now)\n            assert False, message\n        except InvalidToken as e:\n            assert str(e) == message, (str(e), message)\n    assert decode_jwt(token, key, now=2005, leeway=10)[\"sub\"] == \"42\"\n    try:\n        decode_jwt(token, b\"other\", now=1500)\n        assert False\n    except InvalidToken:\n        pass")

### Exercise: Roles, permissions and ownership

Write `allowed(user, action, resource=None)`: `user` is a dict with `"id"` and `"roles"`; `ROLES` maps each role to a set of permissions such as `"orders:read"`. A user may do an action if any of their roles grants it, **or** if the permission is an `:own` variant (`"orders:edit:own"` grants `"orders:edit"`) and `resource["owner"] == user["id"]`. Unknown roles grant nothing (deny by default). Then a decorator `requires(action, load=None)` for miniflask views: it reads the user from `g.user` (answering `401` with `{"error": "log in first"}` if it's `None`), loads the resource with `load(**view_args)` if given, and answers `403` with `{"error": "not allowed"}` unless `allowed(...)`.

In [ ]:
import functools
from miniflask import Flask, g, request

ROLES = {
    "reader": {"orders:read:own"},
    "clerk": {"orders:read", "orders:edit:own"},
    "admin": {"orders:read", "orders:edit", "orders:refund"},
}

def allowed(user, action, resource=None):
    ___

def requires(action, load=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_permissions():\n    \"roles plus ownership\"\n    ada, bob, root = {\"id\": 1, \"roles\": [\"reader\"]}, {\"id\": 2, \"roles\": [\"clerk\"]}, {\"id\": 3, \"roles\": [\"admin\", \"ghost\"]}\n    mine, theirs = {\"owner\": 1}, {\"owner\": 2}\n    assert allowed(ada, \"orders:read\", mine) and not allowed(ada, \"orders:read\", theirs) and not allowed(ada, \"orders:read\")\n    assert allowed(bob, \"orders:read\", mine) and allowed(bob, \"orders:edit\", theirs) and not allowed(bob, \"orders:edit\", mine)\n    assert allowed(root, \"orders:refund\") and not allowed(root, \"orders:delete\") and not allowed({\"id\": 9, \"roles\": [\"ghost\"]}, \"orders:read\", mine)\n    from miniflask import Flask, g, request\n    app = Flask(__name__)\n    ORDERS = {7: {\"id\": 7, \"owner\": 1}}\n    USERS = {\"t1\": ada, \"t2\": bob}\n    @app.before_request\n    def who():\n        g.user = USERS.get(request.headers.get(\"X-Token\"))\n    @app.get(\"/orders/<int:order_id>\")\n    @requires(\"orders:read\", load=lambda order_id: ORDERS[order_id])\n    def read(order_id):\n        return ORDERS[order_id]\n    client = app.test_client()\n    assert client.get(\"/orders/7\").status_code == 401\n    assert client.get(\"/orders/7\", headers={\"X-Token\": \"t1\"}).get_json() == {\"id\": 7, \"owner\": 1}\n    ORDERS[7][\"owner\"] = 5\n    r = client.get(\"/orders/7\", headers={\"X-Token\": \"t1\"})\n    assert r.status_code == 403 and r.get_json() == {\"error\": \"not allowed\"}\n    assert client.get(\"/orders/7\", headers={\"X-Token\": \"t2\"}).status_code == 200")

### Exercise: Slowing down guessers

Write a class `LoginGuard(clock, limit=5, window=900, lockout=900)` with `attempt(username, ok)`: record a login attempt (`ok` says whether the password was right) and return `"ok"`, `"wrong"` or `"locked"`. A user with `limit` failures within the last `window` seconds is locked for `lockout` seconds from the last of them: while locked, **every** attempt returns `"locked"` (even with the right password) and isn't counted. A success clears the user's failures. Usernames are case-insensitive. Also `message(result)`: the text to show: `"Signed in"`, or the same `"Wrong username or password"` for `"wrong"` (so attackers can't tell which was wrong), or `"Too many attempts; try again later"`.

In [ ]:
class LoginGuard:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_guard():\n    \"lockout after repeated failures\"\n    now = [0]\n    guard = LoginGuard(lambda: now[0], limit=3, window=60, lockout=100)\n    assert guard.attempt(\"Ada\", False) == \"wrong\"\n    now[0] = 10\n    assert guard.attempt(\"ada\", True) == \"ok\"\n    for t in (20, 30, 40):\n        now[0] = t\n        result = guard.attempt(\"ADA\", False)\n    assert result == \"wrong\" and guard.attempt(\"ada\", True) == \"locked\"\n    now[0] = 139\n    assert guard.attempt(\"ada\", True) == \"locked\"\n    now[0] = 141\n    assert guard.attempt(\"ada\", True) == \"ok\"\n    for t in (200, 270, 340, 410):\n        now[0] = t\n        assert guard.attempt(\"bob\", False) == \"wrong\", \"failures outside the window don't add up\"\n    assert guard.message(\"wrong\") == \"Wrong username or password\" and guard.message(\"locked\").startswith(\"Too many\")")

### Bug hunt: The same hash for the same password

Passwords are hashed without a salt, so Ada and Bob, who chose the same password, get the same hash, and this prints `True`. Give each user a random 16-byte salt from `secrets` and hash the salt with the password (keeping the salt with the hash), so this prints `False` (and checking still works).

In [ ]:
import hashlib, hmac, secrets

def store(password):
    return "", hashlib.sha256(password.encode()).hexdigest()

def check(password, salt, digest):
    return hmac.compare_digest(hashlib.sha256(bytes.fromhex(salt) + password.encode()).hexdigest(), digest)

ada, bob = store("letmein"), store("letmein")
print(ada[1] == bob[1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"salted hashes\"\n    expect_output(\"False\")\n    salt, digest = ns[\"store\"](\"pw\")\n    assert ns[\"check\"](\"pw\", salt, digest) and not ns[\"check\"](\"px\", salt, digest) and len(bytes.fromhex(salt)) == 16")

### Bug hunt: A token that never expired

The decoder checks the signature but never looks at `exp`, so a token that expired long ago is still accepted and this prints `accepted`. Refuse tokens whose `exp` has passed, so this prints `refused: expired`.

In [ ]:
import base64, hashlib, hmac, json

KEY = b"k"
def b64(raw):
    return base64.urlsafe_b64encode(raw).decode().rstrip("=")
def unb64(text):
    return base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))

def make(payload):
    head = b64(b'{"alg":"HS256","typ":"JWT"}') + "." + b64(json.dumps(payload).encode())
    return head + "." + b64(hmac.new(KEY, head.encode(), hashlib.sha256).digest())

def check(token, now):
    h, p, s = token.split(".")
    if not hmac.compare_digest(b64(hmac.new(KEY, f"{h}.{p}".encode(), hashlib.sha256).digest()), s):
        return "refused: bad signature"
    payload = json.loads(unb64(p))
    return "accepted"

print(check(make({"sub": "1", "exp": 1000}), now=5000))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check exp\"\n    expect_output(\"refused: expired\")")

### Bug hunt: Someone else's order

The endpoint checks that a user is logged in, but not that the order is **theirs**, so Bob can read Ada's order by changing the number in the URL, and this prints `200`. Answer `404` for an order that isn't the user's (so its existence isn't revealed either), so this prints `404`.

In [ ]:
from miniflask import Flask, request, abort

app = Flask(__name__)
ORDERS = {7: {"id": 7, "owner": "ada", "total": 42}}
TOKENS = {"t-ada": "ada", "t-bob": "bob"}

@app.get("/orders/<int:order_id>")
def get_order(order_id):
    user = TOKENS.get(request.headers.get("X-Token"))
    if user is None:
        abort(401)
    if order_id not in ORDERS:
        abort(404)
    return ORDERS[order_id]

print(app.test_client().get("/orders/7", headers={"X-Token": "t-bob"}).status_code)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"object-level authorisation\"\n    expect_output(\"404\")\n    assert ns[\"app\"].test_client().get(\"/orders/7\", headers={\"X-Token\": \"t-ada\"}).status_code == 200")

### Puzzle: A login endpoint

Arrange the lines of a login endpoint that checks the password safely and gives one message for every kind of failure. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: the payload of a JWT is encrypted
q = ___   # the module to use for session ids and salts: "random" or "secrets"
r = ___   # the hmac function that compares secrets in constant time
s = ___   # the status for "you're logged in, but you may not do this"
t = ___   # the JWT claim that holds its expiry time

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is False and q == \"secrets\" and r == \"compare_digest\", \"p, q and r\"\n    assert s == 403 and t == \"exp\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an authenticated API

Write `create_app(key, clock)` (miniflask) with an in-memory user store and these endpoints. `POST /register` takes `{"email", "password"}`: emails are lower-cased and must contain `@` (`422` otherwise), passwords at least 8 characters (`422`), duplicates `409`; it stores a hash from `hash_password` (given, with 200 iterations for speed) and answers `201` with `{"id": N, "email": ...}`, never the hash. `POST /login` checks the password with `verify_password` (given) and answers `200` with `{"token": JWT}` (from `encode_jwt`, given), whose payload has `sub` (the user id), `role` (`"admin"` for the first user registered, `"member"` otherwise) and `exp` (`clock() + 900`); any failure gets `401` with the one message `"wrong email or password"`. `GET /me` requires `Authorization: Bearer TOKEN` (decoded with `decode_jwt`, given; any problem is `401` with `{"error": "invalid token"}`) and returns `{"id", "email", "role"}`. `GET /admin/users` additionally requires the `admin` role (`403` with `{"error": "admins only"}`) and lists every user's `{"id", "email"}`.

In [ ]:
import base64, hashlib, hmac, json, secrets
from miniflask import Flask, request

def _b64(raw):
    return base64.urlsafe_b64encode(raw).decode().rstrip("=")

def _unb64(text):
    return base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))

def hash_password(password, iterations=200):
    salt = secrets.token_bytes(16)
    return f"{iterations}${salt.hex()}${_slow(password.encode(), salt, iterations).hex()}"

def _slow(password, salt, iterations):
    u = hmac.new(password, salt + b"\x00\x00\x00\x01", hashlib.sha256).digest()
    total = int.from_bytes(u, "big")
    for _ in range(iterations - 1):
        u = hmac.new(password, u, hashlib.sha256).digest()
        total ^= int.from_bytes(u, "big")
    return total.to_bytes(32, "big")

def verify_password(password, stored):
    iterations, salt, digest = stored.split("$")
    attempt = _slow(password.encode(), bytes.fromhex(salt), int(iterations))
    return hmac.compare_digest(attempt.hex(), digest)

def encode_jwt(payload, key):
    head = _b64(b'{"alg":"HS256","typ":"JWT"}') + "." + _b64(json.dumps(payload, separators=(",", ":")).encode())
    return head + "." + _b64(hmac.new(key, head.encode(), hashlib.sha256).digest())

def decode_jwt(token, key, now):
    h, p, s = token.split(".")
    if json.loads(_unb64(h)).get("alg") != "HS256":
        raise ValueError("algorithm")
    if not hmac.compare_digest(_b64(hmac.new(key, f"{h}.{p}".encode(), hashlib.sha256).digest()), s):
        raise ValueError("signature")
    payload = json.loads(_unb64(p))
    if now > payload["exp"]:
        raise ValueError("expired")
    return payload

def create_app(key, clock):
    app = Flask(__name__)
    ___
    return app

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_auth_api():\n    \"registration, login, tokens and roles\"\n    now = [1000]\n    client = create_app(b\"test-key\", lambda: now[0]).test_client()\n    r = client.post(\"/register\", json={\"email\": \"Ada@Example.com\", \"password\": \"correct horse\"})\n    assert r.status_code == 201 and r.get_json() == {\"id\": 1, \"email\": \"ada@example.com\"}\n    assert client.post(\"/register\", json={\"email\": \"bob@example.com\", \"password\": \"hunter22\"}).status_code == 201\n    assert client.post(\"/register\", json={\"email\": \"ada@example.com\", \"password\": \"anything8\"}).status_code == 409\n    assert client.post(\"/register\", json={\"email\": \"cy\", \"password\": \"longenough\"}).status_code == 422\n    assert client.post(\"/register\", json={\"email\": \"cy@x\", \"password\": \"short\"}).status_code == 422\n    bad = [client.post(\"/login\", json={\"email\": \"ada@example.com\", \"password\": \"wrong!!!\"}),\n           client.post(\"/login\", json={\"email\": \"nobody@example.com\", \"password\": \"correct horse\"}),\n           client.post(\"/login\", json={})]\n    assert all(r.status_code == 401 and r.get_json() == {\"error\": \"wrong email or password\"} for r in bad)\n    ada = client.post(\"/login\", json={\"email\": \"ADA@example.com\", \"password\": \"correct horse\"}).get_json()[\"token\"]\n    bob = client.post(\"/login\", json={\"email\": \"bob@example.com\", \"password\": \"hunter22\"}).get_json()[\"token\"]\n    assert client.get(\"/me\", headers={\"Authorization\": f\"Bearer {ada}\"}).get_json() == {\"id\": 1, \"email\": \"ada@example.com\", \"role\": \"admin\"}\n    assert client.get(\"/admin/users\", headers={\"Authorization\": f\"Bearer {ada}\"}).get_json() == [{\"id\": 1, \"email\": \"ada@example.com\"}, {\"id\": 2, \"email\": \"bob@example.com\"}]\n    r = client.get(\"/admin/users\", headers={\"Authorization\": f\"Bearer {bob}\"})\n    assert r.status_code == 403 and r.get_json() == {\"error\": \"admins only\"}\n    for headers in [{}, {\"Authorization\": \"Bearer nonsense\"}, {\"Authorization\": f\"Token {ada}\"}, {\"Authorization\": f\"Bearer {ada[:-2]}xx\"}]:\n        r = client.get(\"/me\", headers=headers)\n        assert r.status_code == 401 and r.get_json() == {\"error\": \"invalid token\"}\n    now[0] = 1901\n    assert client.get(\"/me\", headers={\"Authorization\": f\"Bearer {ada}\"}).status_code == 401\n    assert \"hash\" not in str(r.get_json())")

## Warm-up

Hashing the passwords wasn't enough. Ada and Bob have the same hash, so cracking one cracks both, and anyone with the stolen table can hash a list of common passwords once and look every user up instantly. A fast hash like SHA-256 is designed for speed, which is exactly wrong for passwords: attackers can try billions a second. Password storage needs a **salt** and a deliberately **slow** hash.

## Passwords

**Never store passwords**, not even encrypted. Store a value from which the password can be **checked** but not recovered:

1. Generate a random **salt** for each user (`secrets.token_bytes(16)`), so equal passwords get different hashes and precomputed tables are useless.
2. Run a **slow key-derivation function**: **PBKDF2** (repeating HMAC hundreds of thousands of times), **scrypt**, or **Argon2** (the current recommendation, which also uses lots of memory, defeating specialised cracking hardware). `hashlib.pbkdf2_hmac` and `hashlib.scrypt` are in the standard library on most systems; `argon2-cffi` and `bcrypt` are popular packages.
3. Store the algorithm, the cost settings, the salt and the result together, in one string such as `pbkdf2_sha256$600000$SALT$HASH`, so you can raise the cost later and re-hash each user's password when they next log in.
4. To check a login, recompute with the stored salt and settings, and compare with **`hmac.compare_digest`**, which takes the same time however many characters match. A normal `==` stops at the first difference, and the timing can leak how close a guess is.

PBKDF2 itself is short: for each block of output, compute `U1 = HMAC(password, salt + block_number)`, then `U2 = HMAC(password, U1)`, and so on, `iterations` times, XORing all the `U`s together. (This browser's Python has no `hashlib.pbkdf2_hmac`, so today you'll write it, with small iteration counts for speed; real systems use 600,000 or more.)

Also: require reasonable length (at least 8 characters, preferably more) rather than complicated rules, check new passwords against lists of known-breached ones, and never log, email or display a password.

## Staying logged in

After a login, the server must recognise the user's later requests without asking for the password every time. Two common ways:

**Sessions.** The server creates a random **session id** (`secrets.token_urlsafe(32)`), stores `id -> user` (in a database or cache), and sends the id in a cookie (`Set-Cookie: session=...; HttpOnly; Secure; SameSite=Lax`, Week 76 Day 2). Each request's cookie is looked up. Logging out deletes the stored session, which ends it at once. Alternatively, the session data itself can live in the cookie, **signed** with a server-side secret key (an HMAC of the data), so the browser can read but not change it: Flask's default sessions work this way.

**Tokens.** APIs usually send a token in the `Authorization: Bearer ...` header. A **JSON Web Token** (JWT) is three base64url parts joined by dots: a **header** (`{"alg": "HS256", "typ": "JWT"}`), a **payload** of claims (`{"sub": "42", "exp": 1767225600, "role": "admin"}`) and a **signature** (HMAC-SHA256 of the first two parts with a secret key). Anyone can **read** a JWT (it isn't encrypted, so put nothing secret in it); only the key holder can **make** one. Verifying means: recompute the signature and compare in constant time, insist on the algorithm you expect (never accept `"alg": "none"`, a classic attack), and check `exp` (expiry) and `nbf` (not before). Because they're checked without a database, JWTs can't easily be revoked before they expire, so keep them short-lived (minutes) and pair them with a longer-lived **refresh token** that the server does store.

Either way: generate secrets with `secrets`, never `random`; keep keys out of code (Week 79); and set cookies `HttpOnly` and `Secure`.

## Authorisation

Knowing who someone is doesn't say what they may do. Common models:

- **Roles**: users have roles (`reader`, `editor`, `admin`), and each role grants **permissions** (`orders:read`, `orders:refund`). Check permissions, not roles, in code, so roles can change without touching every endpoint.
- **Ownership**: many rules are about **whose** thing it is: you may edit **your own** order. These must be checked against the resource itself, on every request.
- **Deny by default**: an endpoint with no explicit rule should be closed, not open.

The most common real-world access bug (top of the OWASP list, Week 79) is **broken object-level authorisation**: checking that the user is logged in, but not that order 7 is **theirs**, so changing the number in the URL shows someone else's data. Answer `404` rather than `403` when revealing that something exists would itself leak information (yesterday's reading-list challenge did this).

Enforce rules in one place where possible: a decorator or dependency that every protected endpoint uses (Week 60's decorators), with tests that try every endpoint as the wrong user.

## Logins under attack, and OAuth

**Guessing.** Attackers try common passwords against many accounts (**password spraying**), or leaked password lists against yours (**credential stuffing**). Defences: rate-limit login attempts per account and per address, lock or slow an account after repeated failures, offer **multi-factor authentication** (a time-based code or a security key), and give the **same** error message for "no such user" and "wrong password", so attackers can't find out which accounts exist.

**OAuth 2.0 and OpenID Connect.** "Sign in with ..." buttons let another service vouch for the user: your app redirects the user to the identity provider, which authenticates them and redirects back with a short-lived **code** that your server exchanges for tokens. You never see their password. For anything beyond a small project, use a well-reviewed library or service for this; it's easy to get subtly wrong.

## Practice

## On your own computer

Use the well-tested versions of today's pieces:

```bash
pip install "argon2-cffi" pyjwt
```

```python
from argon2 import PasswordHasher
import jwt

ph = PasswordHasher()
stored = ph.hash("correct horse battery staple")       # salt and settings included
ph.verify(stored, "correct horse battery staple")      # raises if wrong

token = jwt.encode({"sub": "42", "exp": 1767225600}, "server-secret", algorithm="HS256")
print(jwt.decode(token, "server-secret", algorithms=["HS256"]))   # checks signature and exp
```

`algorithms=["HS256"]` is what stops the `none` attack: always pass it. Flask's sessions, `flask-login`, and FastAPI's security utilities build on the same ideas.

## Quiz

## Daily challenge

## Before you go

List every place in your projects where a secret (a password, a token, a key) is created, stored, compared or sent, and check each against today's rules. Tomorrow: **deployment**: getting a backend running on a server, configured from its environment, behind HTTPS, with logs and health checks.